In [40]:
import time
from datetime import datetime
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import selenium.common.exceptions as se

date_scraped = datetime.now().strftime("%d-%m-%Y")

amazon_base = "https://www.amazon.com.tr"
amazon_login_page = "https://www.amazon.com.tr/ap/signin?openid.pape.max_auth_age=0&openid.return_to=https%3A%2F%2Fwww.amazon.com.tr%2F%3Fref_%3Dnav_ya_signin&openid.identity=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.assoc_handle=trflex&openid.mode=checkid_setup&openid.claimed_id=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.ns=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0"


def make_driver(detach=True, headless=False):
    chrome_options = webdriver.ChromeOptions()
    if detach:
        chrome_options.add_experimental_option("detach", True)

    # If you want headless:
    if headless:
        chrome_options.add_argument("--headless=new")

    # Stability options
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1400,900")

    # User-Agent (optional)
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

In [49]:
date_scraped = datetime.now().strftime("%d-%m-%Y")

def build_review_url(asin: str, page: int) -> str:
    return (
        f"https://www.amazon.com.tr/product-reviews/{asin}/"
        "ref=cm_cr_dp_d_show_all_btm?ie=UTF8&reviewerType=all_reviews&sortBy=recent"
        f"&pageNumber={page}"
    )

def safe_text(el) -> str:
    try:
        return el.text.strip()
    except Exception:
        return ""

def find_local_and_global_cards(driver):
    """
    Returns (local_cards, global_cards).
    Cards can be li[data-hook=review] or div[data-hook=review] depending on layout.
    """
    container = driver.find_element(By.CSS_SELECTOR, "#cm_cr-review_list")

    # Reviews in this page (works for your HTML)
    cards = container.find_elements(By.CSS_SELECTOR, 'li[data-hook="review"]')

    # Local header exists on TR section
    local_headers = container.find_elements(By.CSS_SELECTOR, 'h3[data-hook="arp-local-reviews-header"]')
    local_text = local_headers[0].text.strip() if local_headers else ""

    # If we don't see the TR local header anymore, assume global section started
    is_local_section = ("Ülke:" in local_text) or ("Türkiye" in local_text)

    if is_local_section:
        local_cards = cards
        global_cards = []
    else:
        local_cards = []
        global_cards = cards

    return local_cards, global_cards

def parse_review_card(card, asin: str, page: int):
    # Title
    try:
        title_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-title"]')
    except Exception:
        title_el = None

    # Rating
    try:
        rating_el = card.find_element(By.CSS_SELECTOR, 'i[data-hook="review-star-rating"] span')
    except Exception:
        rating_el = None

    # Body
    try:
        body_el = card.find_element(By.CSS_SELECTOR, '[data-hook="review-body"]')
    except Exception:
        body_el = None

    # Date (optional)
    try:
        date_el = card.find_element(By.CSS_SELECTOR, 'span[data-hook="review-date"]')
    except Exception:
        date_el = None

    return {
        "asin": asin,
        "page": page,
        "title": safe_text(title_el),
        "rating": safe_text(rating_el),
        "body": safe_text(body_el).replace("\n", " ").strip(),
        "review_date": safe_text(date_el),
        "date_scraped": date_scraped,
    }

def scrape_most_recent_local_reviews(driver, asin: str, target_n: int = 80, max_pages: int = 80, wait_timeout: int = 15):
    """
    - Sayfa başına local+global ayrımı yapar.
    - Global review list görünürse: TR/local bitti -> dur.
    - Local reviewlerden target_n kadar toplar (most recent).
    """
    all_rows = []
    wait = WebDriverWait(driver, wait_timeout)

    for page in range(1, max_pages + 1):
        driver.get(build_review_url(asin, page))

        # En azından bir review container gelsin diye bekle
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-hook="review"]')))
        except se.TimeoutException:
            break

        local_cards, global_cards = find_local_and_global_cards(driver)

        # Global görünmeye başladıysa: local yorumlar bitmiştir -> daha ileri gitme
        if len(global_cards) > 0:
            print("a")
            break

        # Local hiç yoksa zaten bitti
        if len(local_cards) == 0:
            print("b")
            break

        
        for card in local_cards:
            all_rows.append(parse_review_card(card, asin, page))
            if len(all_rows) >= target_n:
                return all_rows[:target_n]

        time.sleep(0.6)  # küçük throttle

    return all_rows[:target_n]

In [45]:
# ----------------- USAGE -----------------
driver = make_driver(detach=True, headless=False)
driver.get(amazon_login_page)

In [54]:
product_ids = ["B07RFD8YZJ"]

all_data = []
for asin in product_ids:
    rows = scrape_most_recent_local_reviews(driver, asin, target_n=80, max_pages=80)
    all_data.extend(rows)

df = pd.DataFrame(all_data)
df.to_csv("amazon_reviews_tr_local_only.csv", index=False, encoding="utf-8-sig")
df.head()

,asin,page,title,rating,body,review_date,date_scraped
0,B07RFD8YZJ,1,Klasik kahvenizden vaz geçmeyeniz.,,Tadı bana hitap etmedi.Çok kavrulmuş ve kekrem...,Türkiye’de 12 Şubat 2026 tarihinde değerlendir...,17-02-2026
1,B07RFD8YZJ,1,.,,Acayip acı içemedik,Türkiye’de 11 Şubat 2026 tarihinde değerlendir...,17-02-2026
2,B07RFD8YZJ,1,Hafif,,İçimi hafif güzel,Türkiye’de 10 Şubat 2026 tarihinde değerlendir...,17-02-2026
3,B07RFD8YZJ,1,Sert içim kadifemsi değil,,Biraz sert bir kahve idi ben aldığımda. İçerik...,Türkiye’de 6 Şubat 2026 tarihinde değerlendirildi,17-02-2026
4,B07RFD8YZJ,1,Beğenmedim,,Çok ağırdı normalde starbucks kahvelerini seve...,Türkiye’de 26 Ocak 2026 tarihinde değerlendirildi,17-02-2026


In [55]:
df.loc[0]["body"]

'Tadı bana hitap etmedi.Çok kavrulmuş ve kekremsi bir tadı var'